## Environment Setup at NCI

Completing this tutorial at NCI requires that users are members of the following projects:

| Code | Description | Needed for |
|------|-------------|------------|
| dk92 | Data science environment | Loading the PyEarthTools Python modules
| nm47 | Training environment | Setting up a working directory for yourself |
| rv74 | Himawari 8 | AutoEncoder examples are based on this | 
| wb00 | (optional) Models data | Accessing ERA5 on-disk data (so you don't need to re-download it) |
| rq0  | (optional) Rainfields3 | Radar-based precipitation data for Australian, may be of interest |

# Introduction to AutoEncoders and Prediction for Weather and Climate

This notebook provides an on-ramp focused on science students, early-career scientists, or later-career scientists who want a quick orientation into the world of machine learning. The focus is instead on the foundational machine learning elements which will give readers the language and concepts they can use to understand those models. A subsequent tutorial series will be developed which incorporates the latest model architectures from recent research, and this initial series will be considered 'required reading'.

This tutorial series includes:


1. (this notebook) A discussion of time-series forecasting (what a data scientist might call a weather prediction model)
2. (this notebook) An introduction into AutoEncoders, starting with why you might want to make one, then going through how to implement one
3. (this notebook) An overview of two data sets - Himawari 8 satellite data, and ERA5 reanalysis data
4. [Data Access Setup at NCI](../DataAccessForWorkshopsNCI.ipynb) - How to check you can access the required data at NCI
5. [Implementing an AutoEncoder](./AutoEncoder_Example.ipynb) on satellite data - first example, using PyEarthTools and PyTorch
6. [Improving Performance of the AutoEncoder](./AutoEncoder_ImprovingResults.ipynb) - a more accurate AutoEncoder plus additional explanation
7. [How to use a TemporalWindow](../TemporalWindowDemo.ipynb) to turn an iterative pipeline into a sequence-to-sequence pipeline
8. [How to extend an AutoEncoder](./FirstPredictionArchitecture.ipynb) to become a forecasting model for satellite data
9. [How to Simplify Training with PyEarthTools](./TrainingWrappers.ipynb) using training wrappers
10. TBD - Exploring alternative architectures for prediction models
12. TBD - Suggestions for experimentation with autoencoders on ERA5 data and extensions into forecasting or prediction

This tutorial might be presented as an integrated workshop, or separated into a 'theory' presentation and a 'practical' hands-on session to follow. For those working on NCI, the bottom of this tutorial includes a list of project codes you will need to join. If you are not working at NCI, you will need to download some data samples from https://geonetwork.nci.org.au/geonetwork/srv/eng/catalog.search#/metadata/f5110_0462_0368_6988. The Thredds server can be used to access sample data.

## Introduction to Time Series Forecasting

This content may seem obvious, but in this tutorial eventually want to make a prediction of future conditions, based on previously obtained observational data. There are a lot of model types which do not fall in this category, including re-analysis, downscaling and feature detection (e.g. detecting cold fronts, rain bands or cyclones) within data. We need to introduce specific language to describe the components of this modelling challenge.

A lot of time series forecasting outside of science is done on one-dimensional data (e.g. a single weather station), or on non-physical data (e.g. a spreadsheet of train passenger numbers by time of day). Time-series data in Earth System data may be done on one dimensional data, a graph of data (e.g. a collection of weather stations, with known locations), a grid of data (like a temperature map) or a volume of data (like a stack of maps, or a cube, based on latitude, longitude and height). Gridded data may refer to 2d data or a volume of data so long as it has clear spatial coordinates. 

A lot of the time, the observations are taken under significant uncertainty. This could be due to physical sensor characteristics or positional uncertainty. As such, there are statistical models, machine learning models and physical models, whose focus is on establishing initial conditions in some fashion. 3d VAR and 4d VAR are two methods grounded in physical science for performing this function. A lot of the time in physical modelling, we will be dealing with a coordinate system of some kind, such as lat, lon, height and time. 

Training such a time-series model (regardless of coordinate structure) involves taking a large archive of historical data, and repeatedly presenting sequences of data to a machine learning model to form the training process. It is common, but by no means the only (or even best) practise to do that by walking through the data from the oldest to the most recent data. There is a lot more that could be said around the biases and issues that could be present in taking a simple approach to model training data, but it is out of scope for this introduction. That said, a failure to understand cross-validation and data leakage between test/train/validate splits is a common issue when first training an ML model.

Two essential questions for any time-series models are "how do you gather observations" and "how far in the future do you predict". There is usually some kind of window of previous observations used. They are typically grouped for convenience into "time steps". These time steps could be for example, hourly, daily, monthly, per-minute, or even more frequently. The model's view of the world is a view of these time steps, and all observations at a given time step are taken as simulteneous. 

```mermaid
flowchart LR

P[t-n...] --> A[t-1] --> B[t=0 ] --> | prediction | C[t+1] --> D[t+1] --> E[...t+n]
```


PyEarthTools makes all of these concepts easy to work with.


## Introduction to AutoEncoders

An AutoEncoder has the odd purpose of trying to reproduce its inputs. That is to say, it optimises for $x = \text{decode}(\text{encode}(x))$ . You can find some great general introductory material at [https://en.wikipedia.org/wiki/Autoencoder](https://en.wikipedia.org/wiki/Autoencoder) and this arXiv paper by [Michelucci (2022)](https://arxiv.org/pdf/2201.03898).

```mermaid
flowchart LR

A[t=0] --> B[encode inputs] --> | latent state | C[decode latent state] --> D[reconstruction of t=0]
```

If you're thinking " ... but why?" then don't worry, you're not alone. There are a few reasons, from the immediate to the longer-term:

1. The encoded state is smaller - it is a dimensional reduction - compared to the original data.
2. The language involved - encoder, latent state, decoder - and the key concepts - training, loss, error, validation - apply equally to more complex architectures which are easier to then understand with a solid grounding
3. Varying the semantics to be a *variational* autoencoder, paves the way towards generative modelling
4. Constraining the bottleneck (e.g. very small vs large) lets you explore the feature space of your science problem
5. Varying the design (i.e. sparse inputs to dense outputs) can pave the way towards data assimilation and downscaling applications
6. Varying the time steps (i.e. t=0 -> t+1) can pave the way towards forecasting and prediction models
7. Varying the architecture of the encoder and decoder lets you examine the performance of alternative network designs
8. If you can't get the errors out of an autoencoder, you can't isolate prediction errors from reconstruction errors

The tutorial [AutoEncoder Example](./AutoEncoder_Example.ipynb) demonstrates how to develop an AutoEncoder in PyEarthTools. The initial tutorial deliberately uses an under-performing network design, in order to focus on core concepts, to allow the demonstration of how this approach is useful in isolating errors, and also to demonstrate how to improve the design to achieve the required goals. The intention is to reveal not only how to build a successful neural network, but also the diagnosis, debugging and development process required to solve problems. An improved-performance example is provided in [AutoEncoder_ImprovingResults](AutoEncoder_ImprovingResults.ipynb).






